# NetOps Analysis — Cost & Quality Governance, Capacity Planning

Data-mining walkthrough over the synthetic CDN/edge-network dataset. Mirrors the kind of
analysis an AI/LLM Data Application team runs: cost decomposition, streaming-quality anomaly
detection, and resource-capacity forecasting. The LLM layer (text-to-SQL and indicator
narration) lives in the `src/` modules and the Streamlit app.

**Prereq:** run `python data/generate_data.py` first to create `data/netops.db`.

In [1]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
from src import db, analysis
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print(db.schema_description())

## 1. Traffic & quality overview

In [2]:
tel = db.run_query('SELECT date, pop_id, peak_traffic_gbps, rebuffer_ratio_pct FROM telemetry')
tel['date'] = pd.to_datetime(tel['date'])
pivot = tel.pivot_table(index='date', columns='pop_id', values='peak_traffic_gbps')
pivot.plot(figsize=(11,4), legend=False, title='Daily peak traffic by PoP (Gbps)'); plt.tight_layout(); plt.show()

DatabaseError: Execution failed on sql 'SELECT date, pop_id, peak_traffic_gbps, rebuffer_ratio_pct FROM telemetry': no such table: telemetry

## 2. Cost governance — where is the money going, and what changed?
Decompose the total-bill change between the first and last month by PoP and component.

In [ ]:
months = db.run_query('SELECT DISTINCT month FROM costs ORDER BY month')['month'].tolist()
dec = analysis.decompose_cost_change(months[0], months[-1])
print(f'Net bill change {months[0]} -> {months[-1]}: ${dec.delta_usd.sum():,.0f}')
top = dec.head(10).copy()
top['driver'] = top['pop_id'] + ' . ' + top['component'].str.replace('_cost_usd','').str.replace('_amort_usd','')
ax = top.plot.barh(x='driver', y='delta_usd', figsize=(9,5), legend=False, title='Top cost-change drivers (USD)')
ax.invert_yaxis(); plt.tight_layout(); plt.show()
top[['pop_id','component','delta_usd']]

## 3. Quality governance — anomaly detection
Robust z-score (median/MAD) flags abnormal streaming-quality days per PoP.

In [ ]:
an = analysis.quality_anomalies('rebuffer_ratio_pct', z_thresh=3.0)
print(f'{len(an)} anomalous PoP-days')
an.head(10)

## 4. Capacity planning — forecast utilisation & days-to-capacity

In [ ]:
cf = analysis.capacity_forecast(horizon_days=30)
cf.head(12)

## 5. LLM indicator narration (optional — needs an API key)
Turn the cost decomposition into a leadership-style governance briefing. Requires
`ANTHROPIC_API_KEY` or `OPENAI_API_KEY` in your environment / `.env`.

In [ ]:
try:
    from src.llm import LLMClient
    print(analysis.narrate_indicator(f'Total-bill change {months[0]}->{months[-1]}', dec, LLMClient()))
except Exception as e:
    print('Skipped (no LLM key configured):', e)